<a href="https://colab.research.google.com/github/luandarodrigues/luandarodrigues/blob/main/IMPULSOGOV_%E2%80%94_PRIORIZA%C3%87%C3%83O_DOS_EST%C3%81GIOS_FORMATIVOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ============================================================
# IMPULSOGOV — PRIORIZAÇÃO DOS ESTÁGIOS FORMATIVOS
# Versão final v2 — Google Colab
# ============================================================

import pandas as pd
import numpy as np
import re
import unicodedata
import os
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)


# ============================================================
# 1. CONFIGURAÇÕES DOS ARQUIVOS
# ============================================================

ARQUIVO_CONTATOS = "/content/contatos.csv"
ARQUIVO_CHATBOT = "/content/respostas_chatbot.csv"
ARQUIVO_SUPERVISOES = "/content/supervisoes_registros.csv"

arquivos_necessarios = {
    "contatos": ARQUIVO_CONTATOS,
    "respostas_chatbot": ARQUIVO_CHATBOT,
    "supervisoes_registros": ARQUIVO_SUPERVISOES,
}

print("Checando arquivos necessários:\n")

for nome, caminho in arquivos_necessarios.items():
    if os.path.exists(caminho):
        print(f"{nome}: OK -> {caminho}")
    else:
        print(f"{nome}: NÃO ENCONTRADO -> {caminho}")

faltantes = [
    nome
    for nome, caminho in arquivos_necessarios.items()
    if not os.path.exists(caminho)
]

if faltantes:
    raise FileNotFoundError(
        "Arquivos faltando no Colab: "
        + ", ".join(faltantes)
        + "\n\nFaça upload dos CSVs no painel lateral do Colab e rode novamente."
    )


# ============================================================
# 2. PARÂMETROS DE NEGÓCIO
# ============================================================

DURACAO_ESTAGIO_SEMANAS = 12

MIN_ACOLHIMENTOS_CERTIFICACAO = 4

MIN_PRESENCA_SUPERVISAO = 0.75
MIN_SUPERVISOES_CERTIFICACAO = 9  # 75% de 12 supervisões

DIAS_SEM_RESPOSTA_ALERTA = 14

PHQ9_SEVERO = 20

# A base real de supervisão usa requisitos AIP com escala 1 a 4.
# Considero adequado quando a resposta é 3 ou 4.
NOTA_REQUISITO_AIP_ADEQUADA = 3

MEDIA_AIP_BAIXA_1A4 = 3.0

PERC_REQUISITOS_ADEQUADOS_BAIXO = 0.75


# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def remover_acentos(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])

    return texto


def normalizar_texto(texto):
    texto = remover_acentos(texto).lower().strip()
    texto = re.sub(r"\s+", " ", texto)

    return texto


def padronizar_colunas(df):
    df = df.copy()
    df.columns = [
        normalizar_texto(c).replace(" ", "_")
        for c in df.columns
    ]

    return df


def extrair_numero(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).replace(",", ".")
    match = re.search(r"-?\d+(\.\d+)?", texto)

    if match:
        return float(match.group(0))

    return np.nan


def parse_data_flex(valor):
    if pd.isna(valor):
        return pd.NaT

    return pd.to_datetime(valor, errors="coerce", dayfirst=True)


def ler_csv(caminho):
    tentativas = [
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ",", "encoding": "latin1"},
        {"sep": ";", "encoding": "latin1"},
    ]

    ultimo_erro = None

    for params in tentativas:
        try:
            df = pd.read_csv(caminho, **params)

            if df.shape[1] > 1:
                return df

        except Exception as e:
            ultimo_erro = e

    raise ultimo_erro


def resumo_base(nome, df):
    print("\n" + "=" * 80)
    print(f"BASE: {nome}")
    print(f"Linhas: {df.shape[0]:,}")
    print(f"Colunas: {df.shape[1]:,}")
    print("Colunas:")
    print(list(df.columns))
    print("=" * 80)


def validar_colunas(df, esperadas, nome_base):
    faltantes = esperadas - set(df.columns)

    if faltantes:
        raise ValueError(
            f"A base {nome_base} está sem as colunas: {sorted(faltantes)}"
        )


# ============================================================
# 4. LEITURA DAS BASES
# ============================================================

contatos = ler_csv(ARQUIVO_CONTATOS)
chatbot = ler_csv(ARQUIVO_CHATBOT)
supervisoes = ler_csv(ARQUIVO_SUPERVISOES)

resumo_base("contatos", contatos)
resumo_base("respostas_chatbot", chatbot)
resumo_base("supervisoes_registros", supervisoes)

contatos = padronizar_colunas(contatos)
chatbot = padronizar_colunas(chatbot)
supervisoes = padronizar_colunas(supervisoes)


# ============================================================
# 5. VALIDAÇÃO MÍNIMA DE COLUNAS
# ============================================================

colunas_contatos = {
    "contato_id",
    "nome",
    "categoria_profissional",
    "supervisor",
    "turma_supervisao",
    "atualizado_em",
}

colunas_chatbot = {
    "contato_id",
    "sessao_id",
    "pergunta_id",
    "resposta",
    "criado_em",
}

colunas_supervisoes = {
    "formulario_id",
    "turma_nome",
    "supervisao_data",
    "profissional_nome",
    "questao",
    "resposta",
    "criado_em",
}

validar_colunas(contatos, colunas_contatos, "contatos")
validar_colunas(chatbot, colunas_chatbot, "respostas_chatbot")
validar_colunas(supervisoes, colunas_supervisoes, "supervisoes_registros")


# ============================================================
# 6. PADRONIZAÇÃO DE TIPOS E TEXTOS
# ============================================================

contatos["atualizado_em"] = pd.to_datetime(contatos["atualizado_em"], errors="coerce")
chatbot["criado_em"] = pd.to_datetime(chatbot["criado_em"], errors="coerce")
supervisoes["supervisao_data"] = pd.to_datetime(supervisoes["supervisao_data"], errors="coerce")
supervisoes["criado_em"] = pd.to_datetime(supervisoes["criado_em"], errors="coerce")


# ============================================================
# 6.1 DATA DE REFERÊNCIA ANALÍTICA
# ============================================================
# Usa a última data válida disponível nas bases.
# Isso evita que o resultado mude conforme o dia em que o código for rodado.

datas_referencia_possiveis = []

datas_referencia_possiveis.append(chatbot["criado_em"].max())

datas_supervisao_validas = supervisoes["supervisao_data"].copy()
datas_supervisao_validas = datas_supervisao_validas[
    datas_supervisao_validas <= pd.Timestamp("2030-01-01")
]
datas_referencia_possiveis.append(datas_supervisao_validas.max())

datas_referencia_possiveis.append(contatos["atualizado_em"].max())

DATA_REFERENCIA = max(
    [d for d in datas_referencia_possiveis if pd.notna(d)]
).normalize()

print("\nDATA_REFERENCIA usada na análise:", DATA_REFERENCIA)


# ============================================================
# 6.2 NORMALIZAÇÃO DE TEXTOS
# ============================================================

contatos["nome_norm"] = contatos["nome"].apply(normalizar_texto)

chatbot["pergunta_id_norm"] = chatbot["pergunta_id"].apply(normalizar_texto)
chatbot["resposta_norm"] = chatbot["resposta"].apply(normalizar_texto)

supervisoes["profissional_nome_norm"] = supervisoes["profissional_nome"].apply(normalizar_texto)
supervisoes["questao_norm"] = supervisoes["questao"].apply(normalizar_texto)
supervisoes["resposta_norm"] = supervisoes["resposta"].apply(normalizar_texto)


# ============================================================
# 7. TRATAMENTO DE CONTATOS DUPLICADOS
# ============================================================

print("\nDeduplicação de contatos")
print("Linhas antes:", contatos.shape[0])
print("Contato_id únicos:", contatos["contato_id"].nunique())

contatos_atual = (
    contatos
    .sort_values("atualizado_em")
    .drop_duplicates(subset=["contato_id"], keep="last")
    .reset_index(drop=True)
)

print("Linhas depois:", contatos_atual.shape[0])


# ============================================================
# 8. INSPEÇÃO DAS PERGUNTAS
# ============================================================

print("\nPerguntas mais frequentes no chatbot:")
display(chatbot["pergunta_id"].value_counts().head(40))

print("\nQuestões mais frequentes nas supervisões:")
display(supervisoes["questao"].value_counts().head(40))


# ============================================================
# 9. MÉTRICAS DO CHATBOT
# ============================================================

def calcular_metricas_chatbot(chatbot):
    df = chatbot.copy()

    base = (
        df.groupby("contato_id")
        .agg(
            interacoes_chatbot=("contato_id", "size"),
            sessoes_chatbot=("sessao_id", "nunique"),
            primeira_resposta_chatbot=("criado_em", "min"),
            ultima_resposta_chatbot=("criado_em", "max"),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # Acolhimentos
    # ------------------------------------------------------------
    # Regra: conta sessões com qual_atendimento válido
    # ou data_atendimento válida.
    # Evita somar números de menus, status e campos administrativos.

    qual = df[df["pergunta_id_norm"] == "qual_atendimento"].copy()

    valores_qual_validos = [
        "1o atendimento",
        "1º atendimento",
        "1 atendimento",
        "primeiro atendimento",
        "2o atendimento",
        "2º atendimento",
        "2 atendimento",
        "segundo atendimento",
        "3o atendimento",
        "3º atendimento",
        "3 atendimento",
        "terceiro atendimento",
        "4o atendimento",
        "4º atendimento",
        "4 atendimento",
        "quarto atendimento",
        "atendimento extra",
        "extra",
        "1",
        "2",
        "3",
        "4",
    ]

    qual["qual_valido"] = qual["resposta_norm"].isin(valores_qual_validos)

    data_at = df[df["pergunta_id_norm"] == "data_atendimento"].copy()
    data_at["data_atendimento_parse"] = data_at["resposta"].apply(parse_data_flex)
    data_at["data_valida"] = data_at["data_atendimento_parse"].notna()

    sessoes_com_acolhimento = pd.concat(
        [
            qual.loc[qual["qual_valido"], ["contato_id", "sessao_id"]],
            data_at.loc[data_at["data_valida"], ["contato_id", "sessao_id"]],
        ],
        ignore_index=True,
    ).drop_duplicates()

    if sessoes_com_acolhimento.empty:
        acolhimentos = pd.DataFrame({
            "contato_id": df["contato_id"].unique(),
            "total_acolhimentos_chatbot": 0,
        })
    else:
        acolhimentos = (
            sessoes_com_acolhimento
            .groupby("contato_id")
            .agg(total_acolhimentos_chatbot=("sessao_id", "nunique"))
            .reset_index()
        )

    # Tipos de atendimento
    qual_validos = qual[qual["qual_valido"]].copy()

    if not qual_validos.empty:
        tipos_atendimento = (
            qual_validos
            .pivot_table(
                index="contato_id",
                columns="resposta_norm",
                values="sessao_id",
                aggfunc="nunique",
                fill_value=0,
            )
            .reset_index()
        )

        tipos_atendimento.columns = [
            "contato_id" if c == "contato_id"
            else f"qtd_{str(c).replace(' ', '_').replace('º', 'o')}"
            for c in tipos_atendimento.columns
        ]
    else:
        tipos_atendimento = pd.DataFrame({"contato_id": df["contato_id"].unique()})

    # ------------------------------------------------------------
    # PHQ-9
    # ------------------------------------------------------------

    phq = df[df["pergunta_id_norm"] == "phq9"].copy()
    phq["phq9"] = pd.to_numeric(
        phq["resposta"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

    phq = phq[phq["phq9"].between(0, 27)]

    if not phq.empty:
        metricas_phq = (
            phq.groupby("contato_id")
            .agg(
                qtd_phq9_registrados=("phq9", "count"),
                max_phq9=("phq9", "max"),
                media_phq9=("phq9", "mean"),
                casos_phq9_severo=("phq9", lambda s: int((s >= PHQ9_SEVERO).sum())),
            )
            .reset_index()
        )
    else:
        metricas_phq = pd.DataFrame({"contato_id": df["contato_id"].unique()})

    # ------------------------------------------------------------
    # Escala de suicídio
    # ------------------------------------------------------------

    suicidio = df[df["pergunta_id_norm"] == "escala_suicidio_respostas"].copy()

    if not suicidio.empty:
        def flag_suicidio(resp):
            r = normalizar_texto(resp)

            termos_alerta = [
                "sim",
                "sim para as duas",
                "apenas p/ pergunta 2",
                "apenas para pergunta 2",
                "pergunta 2",
                "ideacao",
                "ideação",
            ]

            if any(t in r for t in termos_alerta):
                return 1

            return 0

        suicidio["alerta_suicidio"] = suicidio["resposta"].apply(flag_suicidio)

        metricas_suicidio = (
            suicidio.groupby("contato_id")
            .agg(
                registros_escala_suicidio=("alerta_suicidio", "count"),
                alerta_suicidio=("alerta_suicidio", "max"),
            )
            .reset_index()
        )
    else:
        metricas_suicidio = pd.DataFrame({"contato_id": df["contato_id"].unique()})

    # ------------------------------------------------------------
    # Encaminhamento
    # ------------------------------------------------------------

    encaminhamento = df[
        df["pergunta_id_norm"].isin(
            [
                "usuario_fora_do_perfil",
                "justificativa_situacao",
                "justificativa_situação",
                "estrategias_rastreamento_outro",
                "estratégias_rastreamento_outro",
            ]
        )
    ].copy()

    if not encaminhamento.empty:
        encaminhamento["encaminhamento_flag"] = encaminhamento["resposta_norm"].str.contains(
            "encaminhamento|encaminhado|caps|raps|servico especializado|serviço especializado|psiquiatria|psicologia",
            regex=True,
            na=False,
        ).astype(int)

        metricas_enc = (
            encaminhamento.groupby("contato_id")
            .agg(
                encaminhamentos_registrados=("encaminhamento_flag", "sum"),
                algum_encaminhamento=("encaminhamento_flag", "max"),
            )
            .reset_index()
        )
    else:
        metricas_enc = pd.DataFrame({"contato_id": df["contato_id"].unique()})

    # ------------------------------------------------------------
    # Justificativas e barreiras
    # ------------------------------------------------------------

    justificativas = df[
        df["pergunta_id_norm"].isin(
            [
                "justificativa_situacao",
                "justificativa_situação",
                "justificativa_detalhes",
            ]
        )
    ].copy()

    if not justificativas.empty:
        justificativas["flag_barreira"] = justificativas["resposta_norm"].str.contains(
            "desistiu|evasao|evadiu|ferias|férias|licenca|licença|sobrecarga|usuario faltou|usuário faltou|nao localizado|não localizado|desmotivacao|desmotivação|sala|internet|agenda",
            regex=True,
            na=False,
        ).astype(int)

        metricas_just = (
            justificativas.groupby("contato_id")
            .agg(
                qtd_justificativas=("resposta", "count"),
                justificativa_barreira=("flag_barreira", "max"),
            )
            .reset_index()
        )
    else:
        metricas_just = pd.DataFrame({"contato_id": df["contato_id"].unique()})

    out = base.merge(acolhimentos, on="contato_id", how="left")
    out = out.merge(tipos_atendimento, on="contato_id", how="left")
    out = out.merge(metricas_phq, on="contato_id", how="left")
    out = out.merge(metricas_suicidio, on="contato_id", how="left")
    out = out.merge(metricas_enc, on="contato_id", how="left")
    out = out.merge(metricas_just, on="contato_id", how="left")

    return out


metricas_chatbot = calcular_metricas_chatbot(chatbot)

print("\nMétricas do chatbot:")
display(metricas_chatbot.head())

print("\nDistribuição de acolhimentos por chatbot:")
display(metricas_chatbot["total_acolhimentos_chatbot"].describe())


# ============================================================
# 10. MÉTRICAS DE SUPERVISÃO
# ============================================================

def calcular_metricas_supervisao(supervisoes):
    df = supervisoes.copy()

    # Remove datas absurdamente futuras.
    df["supervisao_data_original"] = df["supervisao_data"]
    df.loc[df["supervisao_data"] > pd.Timestamp("2030-01-01"), "supervisao_data"] = pd.NaT

    # Chave correta:
    # formulario_id sozinho não é único.
    df["form_key"] = (
        df["formulario_id"].astype(str)
        + " | "
        + df["profissional_nome_norm"].astype(str)
        + " | "
        + df["supervisao_data"].astype(str)
    )

    forms = (
        df[["form_key", "profissional_nome_norm", "supervisao_data", "turma_nome"]]
        .drop_duplicates()
        .copy()
    )

    # ------------------------------------------------------------
    # Presença em supervisão
    # ------------------------------------------------------------
    # Se há motivo para não comparecimento, então faltou.
    # Se não há motivo, presença inferida pelo formulário.

    motivos_falta = df[
        df["questao_norm"] == "por qual motivo o profissional nao compareceu a supervisao?"
    ][["form_key", "resposta_norm"]].copy()

    motivos_falta["faltou_supervisao"] = motivos_falta["resposta_norm"].ne("")

    motivos_falta = (
        motivos_falta
        .groupby("form_key")
        .agg(faltou_supervisao=("faltou_supervisao", "max"))
        .reset_index()
    )

    forms = forms.merge(motivos_falta, on="form_key", how="left")
    forms["faltou_supervisao"] = forms["faltou_supervisao"].fillna(False)
    forms["presente_supervisao"] = np.where(forms["faltou_supervisao"], 0, 1)

    metricas_presenca = (
        forms.groupby("profissional_nome_norm")
        .agg(
            formularios_supervisao=("form_key", "nunique"),
            presencas_supervisao=("presente_supervisao", "sum"),
            faltas_supervisao=("faltou_supervisao", "sum"),
            primeira_supervisao=("supervisao_data", "min"),
            ultima_supervisao=("supervisao_data", "max"),
        )
        .reset_index()
    )

    # ------------------------------------------------------------
    # Atendimento realizado segundo supervisor
    # ------------------------------------------------------------

    q_atendimento = df[
        df["questao_norm"].isin(
            [
                "profissional realizou atendimento?",
                "realizou atendimento",
            ]
        )
    ][["form_key", "profissional_nome_norm", "resposta_norm"]].copy()

    def atendimento_realizado_flag(resp):
        if resp == "sim":
            return 1

        if resp == "nao":
            return 0

        if "faltou" in resp or "evadiu" in resp:
            return 0

        return np.nan

    if not q_atendimento.empty:
        q_atendimento["atendimento_realizado_flag"] = q_atendimento["resposta_norm"].apply(atendimento_realizado_flag)

        metricas_atendimento_sup = (
            q_atendimento.groupby("profissional_nome_norm")
            .agg(
                semanas_com_atendimento_supervisor=("atendimento_realizado_flag", "sum"),
                semanas_avaliadas_atendimento=("atendimento_realizado_flag", "count"),
            )
            .reset_index()
        )
    else:
        metricas_atendimento_sup = pd.DataFrame({"profissional_nome_norm": df["profissional_nome_norm"].unique()})

    # ------------------------------------------------------------
    # Atendimento realizado por usuário na última semana
    # ------------------------------------------------------------

    atendimento_usuario = df[
        df["questao_norm"].str.startswith("atendimento realizado na ultima semana")
    ].copy()

    respostas_validas_atendimento = [
        "rastreamento",
        "1o",
        "1º",
        "2o",
        "2º",
        "3o",
        "3º",
        "4o",
        "4º",
    ]

    if not atendimento_usuario.empty:
        atendimento_usuario["atendimento_usuario_flag"] = atendimento_usuario["resposta_norm"].isin(
            respostas_validas_atendimento
        ).astype(int)

        metricas_atendimento_usuario = (
            atendimento_usuario.groupby("profissional_nome_norm")
            .agg(
                registros_atendimento_usuario=("atendimento_usuario_flag", "sum")
            )
            .reset_index()
        )
    else:
        metricas_atendimento_usuario = pd.DataFrame({"profissional_nome_norm": df["profissional_nome_norm"].unique()})

    # ------------------------------------------------------------
    # Qualidade do AIP por requisitos obrigatórios
    # ------------------------------------------------------------

    requisitos = df[
        df["questao_norm"].str.startswith("requisitos obrigatorios para todos os atendimentos")
    ].copy()

    requisitos["nota_requisito_aip"] = pd.to_numeric(requisitos["resposta"], errors="coerce")
    requisitos = requisitos[requisitos["nota_requisito_aip"].between(1, 4)]

    if not requisitos.empty:
        metricas_aip = (
            requisitos.groupby("profissional_nome_norm")
            .agg(
                itens_aip_avaliados=("nota_requisito_aip", "count"),
                media_requisitos_aip_1a4=("nota_requisito_aip", "mean"),
                menor_requisito_aip_1a4=("nota_requisito_aip", "min"),
                perc_requisitos_adequados=("nota_requisito_aip", lambda s: float((s >= NOTA_REQUISITO_AIP_ADEQUADA).mean())),
            )
            .reset_index()
        )
    else:
        metricas_aip = pd.DataFrame({"profissional_nome_norm": df["profissional_nome_norm"].unique()})

    # ------------------------------------------------------------
    # Motivos de não atendimento
    # ------------------------------------------------------------

    motivos_sem_atendimento = df[
        df["questao_norm"] == "por qual motivo o profissional nao realizou nenhum atendimento na ultima semana?"
    ][["profissional_nome_norm", "resposta_norm"]].copy()

    if not motivos_sem_atendimento.empty:
        motivos_sem_atendimento["flag_barreira_atendimento"] = motivos_sem_atendimento["resposta_norm"].str.contains(
            "usuario nao compareceu|usuário não compareceu|dificuldade|desmotivacao|desmotivação|sobrecarga|recusou|nao localizado|não localizado|sala|ferias|férias|licenca|licença|evasao|evasão|desligado|agenda",
            regex=True,
            na=False,
        ).astype(int)

        metricas_motivos = (
            motivos_sem_atendimento.groupby("profissional_nome_norm")
            .agg(
                qtd_motivos_sem_atendimento=("resposta_norm", lambda s: int((s != "").sum())),
                barreira_atendimento_supervisor=("flag_barreira_atendimento", "max"),
            )
            .reset_index()
        )
    else:
        metricas_motivos = pd.DataFrame({"profissional_nome_norm": df["profissional_nome_norm"].unique()})

    out = metricas_presenca.merge(metricas_atendimento_sup, on="profissional_nome_norm", how="left")
    out = out.merge(metricas_atendimento_usuario, on="profissional_nome_norm", how="left")
    out = out.merge(metricas_aip, on="profissional_nome_norm", how="left")
    out = out.merge(metricas_motivos, on="profissional_nome_norm", how="left")

    return out


metricas_supervisao = calcular_metricas_supervisao(supervisoes)

print("\nMétricas de supervisão:")
display(metricas_supervisao.head())

print("\nDistribuição de presenças em supervisão:")
display(metricas_supervisao["presencas_supervisao"].describe())

print("\nDistribuição da média dos requisitos AIP:")
display(metricas_supervisao["media_requisitos_aip_1a4"].describe())


# ============================================================
# 11. CONSOLIDAÇÃO DAS BASES
# ============================================================

base = contatos_atual.copy()

base = base.merge(
    metricas_chatbot,
    on="contato_id",
    how="left"
)

base = base.merge(
    metricas_supervisao,
    left_on="nome_norm",
    right_on="profissional_nome_norm",
    how="left"
)


# ============================================================
# 12. PREENCHIMENTO DE AUSÊNCIAS
# ============================================================

colunas_zero = [
    "interacoes_chatbot",
    "sessoes_chatbot",
    "total_acolhimentos_chatbot",
    "qtd_phq9_registrados",
    "casos_phq9_severo",
    "registros_escala_suicidio",
    "alerta_suicidio",
    "encaminhamentos_registrados",
    "algum_encaminhamento",
    "qtd_justificativas",
    "justificativa_barreira",
    "formularios_supervisao",
    "presencas_supervisao",
    "faltas_supervisao",
    "semanas_com_atendimento_supervisor",
    "semanas_avaliadas_atendimento",
    "registros_atendimento_usuario",
    "itens_aip_avaliados",
    "media_requisitos_aip_1a4",
    "menor_requisito_aip_1a4",
    "perc_requisitos_adequados",
    "qtd_motivos_sem_atendimento",
    "barreira_atendimento_supervisor",
]

for col in colunas_zero:
    if col in base.columns:
        base[col] = base[col].fillna(0)

for col in [
    "ultima_resposta_chatbot",
    "ultima_supervisao",
    "primeira_resposta_chatbot",
    "primeira_supervisao",
]:
    if col in base.columns:
        base[col] = pd.to_datetime(base[col], errors="coerce")

base["dias_sem_resposta_chatbot"] = (DATA_REFERENCIA - base["ultima_resposta_chatbot"]).dt.days
base["dias_sem_resposta_chatbot"] = base["dias_sem_resposta_chatbot"].fillna(999)

base["dias_sem_supervisao"] = (DATA_REFERENCIA - base["ultima_supervisao"]).dt.days
base["dias_sem_supervisao"] = base["dias_sem_supervisao"].fillna(999)

# Corrige eventuais dias negativos por datas futuras residuais.
base.loc[base["dias_sem_supervisao"] < 0, "dias_sem_supervisao"] = np.nan


# ============================================================
# 13. ESTIMATIVA DE INÍCIO DO ESTÁGIO
# ============================================================

def estimar_inicio_estagio(row):
    datas = []

    for col in ["primeira_resposta_chatbot", "primeira_supervisao", "atualizado_em"]:
        if col in row.index and pd.notna(row[col]):
            datas.append(row[col])

    if len(datas) == 0:
        return DATA_REFERENCIA - pd.Timedelta(days=7)

    return min(datas)


base["inicio_estagio_estimado"] = base.apply(estimar_inicio_estagio, axis=1)

base["semanas_decorridas"] = (
    ((DATA_REFERENCIA - base["inicio_estagio_estimado"]).dt.days // 7) + 1
)

base["semanas_decorridas"] = base["semanas_decorridas"].clip(
    lower=1,
    upper=DURACAO_ESTAGIO_SEMANAS
)


# ============================================================
# 14. INDICADORES DERIVADOS
# ============================================================

base["acolhimentos_esperados_ate_agora"] = (
    MIN_ACOLHIMENTOS_CERTIFICACAO
    * base["semanas_decorridas"]
    / DURACAO_ESTAGIO_SEMANAS
)

base["supervisoes_esperadas_ate_agora"] = base["semanas_decorridas"]

base["presenca_esperada_minima_ate_agora"] = (
    MIN_PRESENCA_SUPERVISAO
    * base["supervisoes_esperadas_ate_agora"]
)

base["gap_acolhimentos"] = (
    base["acolhimentos_esperados_ate_agora"]
    - base["total_acolhimentos_chatbot"]
).clip(lower=0)

base["gap_presencas"] = (
    base["presenca_esperada_minima_ate_agora"]
    - base["presencas_supervisao"]
).clip(lower=0)

base["taxa_presenca_supervisao"] = (
    base["presencas_supervisao"]
    / base["supervisoes_esperadas_ate_agora"].replace(0, np.nan)
).clip(upper=1).fillna(0)

base["taxa_resposta_chatbot"] = (
    base["sessoes_chatbot"]
    / base["semanas_decorridas"].replace(0, np.nan)
).clip(upper=1).fillna(0)

# Regra final v2:
# presença capada em 12 apenas para certificação.
base["presencas_supervisao_certificacao"] = (
    base["presencas_supervisao"]
    .clip(upper=DURACAO_ESTAGIO_SEMANAS)
)

base["taxa_presenca_supervisao_certificacao"] = (
    base["presencas_supervisao_certificacao"]
    / DURACAO_ESTAGIO_SEMANAS
)


# ============================================================
# 15. SCORE DE PRIORIZAÇÃO
# ============================================================

# ------------------------------------------------------------
# 15.1 Risco de evasão dos acolhimentos — até 35 pontos
# ------------------------------------------------------------

base["risco_acolhimento_score"] = (
    20
    * (
        base["gap_acolhimentos"]
        / base["acolhimentos_esperados_ate_agora"].replace(0, np.nan)
    )
).clip(upper=20).fillna(0)

base["flag_sem_resposta_14d"] = (
    base["dias_sem_resposta_chatbot"] >= DIAS_SEM_RESPOSTA_ALERTA
)

base.loc[
    base["flag_sem_resposta_14d"],
    "risco_acolhimento_score"
] += 10

base["flag_barreira_atendimento"] = (
    (base["justificativa_barreira"] > 0)
    | (base["barreira_atendimento_supervisor"] > 0)
)

base.loc[
    base["flag_barreira_atendimento"],
    "risco_acolhimento_score"
] += 5

base["risco_acolhimento_score"] = base["risco_acolhimento_score"].clip(upper=35)


# ------------------------------------------------------------
# 15.2 Risco de evasão das supervisões — até 30 pontos
# ------------------------------------------------------------

base["risco_supervisao_score"] = (
    25
    * (
        base["gap_presencas"]
        / base["presenca_esperada_minima_ate_agora"].replace(0, np.nan)
    )
).clip(upper=25).fillna(0)

base["flag_sem_supervisao_registrada"] = (
    (base["semanas_decorridas"] >= 3)
    & (base["formularios_supervisao"] == 0)
)

base.loc[
    base["flag_sem_supervisao_registrada"],
    "risco_supervisao_score"
] += 5

base["risco_supervisao_score"] = base["risco_supervisao_score"].clip(upper=30)


# ------------------------------------------------------------
# 15.3 Má aplicação do AIP — até 25 pontos
# ------------------------------------------------------------

base["risco_aip_score"] = 0

base.loc[
    (base["itens_aip_avaliados"] > 0)
    & (base["media_requisitos_aip_1a4"] < MEDIA_AIP_BAIXA_1A4),
    "risco_aip_score"
] += 13

base.loc[
    (base["itens_aip_avaliados"] > 0)
    & (base["perc_requisitos_adequados"] < PERC_REQUISITOS_ADEQUADOS_BAIXO),
    "risco_aip_score"
] += 9

base["flag_sem_avaliacao_aip"] = (
    (base["semanas_decorridas"] >= 4)
    & (base["itens_aip_avaliados"] == 0)
)

base.loc[
    base["flag_sem_avaliacao_aip"],
    "risco_aip_score"
] += 3

base["risco_aip_score"] = base["risco_aip_score"].clip(upper=25)


# ------------------------------------------------------------
# 15.4 Alerta clínico / encaminhamento — até 10 pontos
# ------------------------------------------------------------

base["alerta_clinico_score"] = 0

base.loc[
    base["casos_phq9_severo"] > 0,
    "alerta_clinico_score"
] += 4

base.loc[
    base["alerta_suicidio"] > 0,
    "alerta_clinico_score"
] += 4

base["flag_phq9_severo_sem_encaminhamento"] = (
    (base["casos_phq9_severo"] > 0)
    & (base["encaminhamentos_registrados"] == 0)
)

base.loc[
    base["flag_phq9_severo_sem_encaminhamento"],
    "alerta_clinico_score"
] += 2

base["alerta_clinico_score"] = base["alerta_clinico_score"].clip(upper=10)

base["alerta_clinico_aberto"] = (
    (base["casos_phq9_severo"] > 0)
    | (base["alerta_suicidio"] > 0)
    | (base["flag_phq9_severo_sem_encaminhamento"])
)


# ------------------------------------------------------------
# 15.5 Score final
# ------------------------------------------------------------

base["score_prioridade"] = (
    base["risco_acolhimento_score"]
    + base["risco_supervisao_score"]
    + base["risco_aip_score"]
    + base["alerta_clinico_score"]
).round(1)


def classificar_prioridade(score):
    if score >= 70:
        return "crítico"

    if score >= 45:
        return "alto"

    if score >= 25:
        return "médio"

    return "baixo"


base["nivel_prioridade"] = base["score_prioridade"].apply(classificar_prioridade)

base["flag_risco_evasao_acolhimentos"] = base["risco_acolhimento_score"] >= 18
base["flag_risco_evasao_supervisoes"] = base["risco_supervisao_score"] >= 15
base["flag_ma_aplicacao_aip"] = base["risco_aip_score"] >= 12


def motivo_principal(row):
    scores = {
        "Risco de evasão dos acolhimentos": row["risco_acolhimento_score"],
        "Risco de evasão das supervisões": row["risco_supervisao_score"],
        "Má aplicação do AIP": row["risco_aip_score"],
        "Alerta clínico/encaminhamento": row["alerta_clinico_score"],
    }

    return max(scores, key=scores.get)


base["motivo_principal_priorizacao"] = base.apply(motivo_principal, axis=1)


def acao_sugerida(row):
    if row["nivel_prioridade"] == "crítico":
        return "Contato ativo com profissional e discussão com supervisor na semana"

    if row["flag_phq9_severo_sem_encaminhamento"]:
        return "Validar fluxo de encaminhamento para caso com PHQ-9 severo"

    if row["alerta_clinico_aberto"]:
        return "Revisar alerta clínico e confirmar conduta/encaminhamento registrado"

    if row["flag_ma_aplicacao_aip"]:
        return "Revisar aplicação do AIP em supervisão e indicar reforço formativo"

    if row["flag_risco_evasao_supervisoes"]:
        return "Checar barreira de agenda, presença e vínculo com turma de supervisão"

    if row["flag_risco_evasao_acolhimentos"]:
        return "Investigar motivo de não realização dos acolhimentos"

    return "Manter acompanhamento de rotina"


base["acao_sugerida"] = base.apply(acao_sugerida, axis=1)


# ============================================================
# 16. CERTIFICAÇÃO — VERSÃO FINAL V2
# ============================================================

base["apto_certificacao"] = (
    (base["total_acolhimentos_chatbot"] >= MIN_ACOLHIMENTOS_CERTIFICACAO)
    & (base["presencas_supervisao_certificacao"] >= MIN_SUPERVISOES_CERTIFICACAO)
    & (~base["flag_ma_aplicacao_aip"])
)

base["status_estagio"] = np.select(
    [
        base["apto_certificacao"] & base["alerta_clinico_aberto"],
        base["apto_certificacao"],
        base["nivel_prioridade"].eq("crítico"),
        base["nivel_prioridade"].eq("alto"),
        base["nivel_prioridade"].eq("médio"),
    ],
    [
        "Apto com alerta clínico",
        "Apto à certificação",
        "Crítico",
        "Atenção alta",
        "Atenção média",
    ],
    default="Em acompanhamento"
)


# ============================================================
# 17. TABELA FINAL DE PRIORIZAÇÃO
# ============================================================

colunas_saida = [
    "contato_id",
    "nome",
    "categoria_profissional",
    "supervisor",
    "turma_supervisao",

    "inicio_estagio_estimado",
    "semanas_decorridas",

    "total_acolhimentos_chatbot",
    "acolhimentos_esperados_ate_agora",
    "gap_acolhimentos",
    "sessoes_chatbot",
    "taxa_resposta_chatbot",
    "dias_sem_resposta_chatbot",

    "formularios_supervisao",
    "presencas_supervisao",
    "presencas_supervisao_certificacao",
    "faltas_supervisao",
    "supervisoes_esperadas_ate_agora",
    "taxa_presenca_supervisao",
    "taxa_presenca_supervisao_certificacao",
    "dias_sem_supervisao",

    "semanas_com_atendimento_supervisor",
    "registros_atendimento_usuario",

    "itens_aip_avaliados",
    "media_requisitos_aip_1a4",
    "menor_requisito_aip_1a4",
    "perc_requisitos_adequados",

    "qtd_phq9_registrados",
    "max_phq9",
    "media_phq9",
    "casos_phq9_severo",
    "alerta_suicidio",
    "encaminhamentos_registrados",

    "risco_acolhimento_score",
    "risco_supervisao_score",
    "risco_aip_score",
    "alerta_clinico_score",
    "score_prioridade",
    "nivel_prioridade",
    "status_estagio",
    "motivo_principal_priorizacao",
    "acao_sugerida",

    "flag_sem_resposta_14d",
    "flag_barreira_atendimento",
    "flag_risco_evasao_acolhimentos",
    "flag_risco_evasao_supervisoes",
    "flag_ma_aplicacao_aip",
    "flag_phq9_severo_sem_encaminhamento",
    "alerta_clinico_aberto",
    "apto_certificacao",
]

colunas_saida = [c for c in colunas_saida if c in base.columns]

tabela_priorizacao = (
    base[colunas_saida]
    .sort_values(
        by=["score_prioridade", "dias_sem_resposta_chatbot"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("\nTabela final v2 de priorização:")
display(tabela_priorizacao.head(30))


# ============================================================
# 18. RESUMOS
# ============================================================

print("\nResumo por nível de prioridade:")
resumo_prioridade = (
    tabela_priorizacao["nivel_prioridade"]
    .value_counts()
    .rename_axis("nivel_prioridade")
    .reset_index(name="qtd_profissionais")
)

display(resumo_prioridade)


print("\nResumo por status do estágio:")
resumo_status = (
    tabela_priorizacao["status_estagio"]
    .value_counts()
    .rename_axis("status_estagio")
    .reset_index(name="qtd_profissionais")
)

display(resumo_status)


print("\nResumo por supervisor:")
resumo_supervisor = (
    tabela_priorizacao
    .groupby("supervisor")
    .agg(
        profissionais=("contato_id", "count"),
        score_medio=("score_prioridade", "mean"),
        criticos=("nivel_prioridade", lambda s: int((s == "crítico").sum())),
        alto_risco=("nivel_prioridade", lambda s: int((s == "alto").sum())),
        medio_risco=("nivel_prioridade", lambda s: int((s == "médio").sum())),
        baixo_risco=("nivel_prioridade", lambda s: int((s == "baixo").sum())),
        aptos_certificacao=("apto_certificacao", "sum"),
        aptos_com_alerta_clinico=("status_estagio", lambda s: int((s == "Apto com alerta clínico").sum())),
        alertas_clinicos_abertos=("alerta_clinico_aberto", "sum"),
        presenca_media_certificacao=("taxa_presenca_supervisao_certificacao", "mean"),
        acolhimentos_media=("total_acolhimentos_chatbot", "mean"),
        media_requisitos_aip=("media_requisitos_aip_1a4", "mean"),
        perc_requisitos_adequados=("perc_requisitos_adequados", "mean"),
    )
    .reset_index()
    .sort_values(
        by=["criticos", "alto_risco", "score_medio"],
        ascending=[False, False, False]
    )
)

display(resumo_supervisor)


print("\nResumo por turma:")
resumo_turma = (
    tabela_priorizacao
    .groupby("turma_supervisao")
    .agg(
        profissionais=("contato_id", "count"),
        score_medio=("score_prioridade", "mean"),
        criticos=("nivel_prioridade", lambda s: int((s == "crítico").sum())),
        alto_risco=("nivel_prioridade", lambda s: int((s == "alto").sum())),
        medio_risco=("nivel_prioridade", lambda s: int((s == "médio").sum())),
        baixo_risco=("nivel_prioridade", lambda s: int((s == "baixo").sum())),
        aptos_certificacao=("apto_certificacao", "sum"),
        aptos_com_alerta_clinico=("status_estagio", lambda s: int((s == "Apto com alerta clínico").sum())),
        alertas_clinicos_abertos=("alerta_clinico_aberto", "sum"),
        presenca_media_certificacao=("taxa_presenca_supervisao_certificacao", "mean"),
        acolhimentos_media=("total_acolhimentos_chatbot", "mean"),
        media_requisitos_aip=("media_requisitos_aip_1a4", "mean"),
        perc_requisitos_adequados=("perc_requisitos_adequados", "mean"),
    )
    .reset_index()
    .sort_values(
        by=["criticos", "alto_risco", "score_medio"],
        ascending=[False, False, False]
    )
)

display(resumo_turma)


# ============================================================
# 19. CHECKS DE QUALIDADE DOS DADOS
# ============================================================

print("\nChecks de qualidade dos dados:")

datas_supervisao_originais = pd.to_datetime(
    supervisoes["supervisao_data"],
    errors="coerce"
)

checks = {
    "data_referencia_analitica": DATA_REFERENCIA,

    "linhas_contatos_original": len(contatos),
    "contatos_unicos_final": len(contatos_atual),
    "contatos_duplicados_removidos": len(contatos) - len(contatos_atual),

    "linhas_chatbot": len(chatbot),
    "linhas_supervisoes": len(supervisoes),

    "formularios_id_unicos": supervisoes["formulario_id"].nunique(),
    "combinacoes_formulario_profissional_data": supervisoes[
        ["formulario_id", "profissional_nome", "supervisao_data"]
    ].drop_duplicates().shape[0],

    "datas_supervisao_futuras_acima_2030": int(
        (datas_supervisao_originais > pd.Timestamp("2030-01-01")).sum()
    ),

    "profissionais_sem_chatbot": int(
        tabela_priorizacao["sessoes_chatbot"].eq(0).sum()
    ),

    "profissionais_sem_supervisao": int(
        tabela_priorizacao["formularios_supervisao"].eq(0).sum()
    ),

    "profissionais_aptos_certificacao": int(
        tabela_priorizacao["apto_certificacao"].sum()
    ),

    "profissionais_aptos_com_alerta_clinico": int(
        tabela_priorizacao["status_estagio"].eq("Apto com alerta clínico").sum()
    ),

    "profissionais_com_alerta_clinico_aberto": int(
        tabela_priorizacao["alerta_clinico_aberto"].sum()
    ),

    "profissionais_criticos": int(
        tabela_priorizacao["nivel_prioridade"].eq("crítico").sum()
    ),
}

checks_df = pd.DataFrame(
    list(checks.items()),
    columns=["check", "valor"]
)

display(checks_df)


# ============================================================
# 20. EXPORTAÇÃO — ARQUIVOS FINAIS V2
# ============================================================

tabela_priorizacao.to_csv(
    "tabela_priorizacao_estagio_formativo_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_supervisor.to_csv(
    "resumo_por_supervisor_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_turma.to_csv(
    "resumo_por_turma_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_prioridade.to_csv(
    "resumo_por_prioridade_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_status.to_csv(
    "resumo_por_status_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

checks_df.to_csv(
    "checks_qualidade_dados_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nArquivos gerados:")
print("- tabela_priorizacao_estagio_formativo_v2.csv")
print("- resumo_por_supervisor_v2.csv")
print("- resumo_por_turma_v2.csv")
print("- resumo_por_prioridade_v2.csv")
print("- resumo_por_status_v2.csv")
print("- checks_qualidade_dados_v2.csv")


# ============================================================
# 21. DOWNLOAD AUTOMÁTICO NO COLAB
# ============================================================

try:
    from google.colab import files

    files.download("tabela_priorizacao_estagio_formativo_v2.csv")
    files.download("resumo_por_supervisor_v2.csv")
    files.download("resumo_por_turma_v2.csv")
    files.download("resumo_por_prioridade_v2.csv")
    files.download("resumo_por_status_v2.csv")
    files.download("checks_qualidade_dados_v2.csv")

except Exception:
    print("\nDownload automático disponível apenas no Google Colab.")

Checando arquivos necessários:

contatos: OK -> /content/contatos.csv
respostas_chatbot: OK -> /content/respostas_chatbot.csv
supervisoes_registros: OK -> /content/supervisoes_registros.csv

BASE: contatos
Linhas: 94
Colunas: 6
Colunas:
['contato_id', 'nome', 'categoria_profissional', 'supervisor', 'turma_supervisao', 'atualizado_em']

BASE: respostas_chatbot
Linhas: 19,279
Colunas: 5
Colunas:
['contato_id', 'sessao_id', 'pergunta_id', 'resposta', 'criado_em']

BASE: supervisoes_registros
Linhas: 22,044
Colunas: 7
Colunas:
['formulario_id', 'turma_nome', 'supervisao_data', 'profissional_nome', 'questao', 'resposta', 'criado_em']

DATA_REFERENCIA usada na análise: 2029-04-29 00:00:00

Deduplicação de contatos
Linhas antes: 94
Contato_id únicos: 77
Linhas depois: 77

Perguntas mais frequentes no chatbot:


,count
pergunta_id,
nome,2616
categoria,2604
status,2569
supervisor,1890
turma_projeto,1267
menu_o_que_gostaria_de_fazer,1264
sigla_usuario,1074
data_atendimento,1046
phq9,1022



Questões mais frequentes nas supervisões:


,count
questao,
Requisitos obrigatórios para todos os atendimentos [Postura ativa e escuta qualificada],1338
Requisitos obrigatórios para todos os atendimentos [Aplicação dos instrumentos no tempo do atendimento],1338
Atendimento realizado na última semana [Usuário 4],1338
Atendimento realizado na última semana [Usuário 1],1338
Atendimento realizado na última semana [Usuário 2],1338
Requisitos obrigatórios para todos os atendimentos [Estratégias de cuidado],1338
Requisitos obrigatórios para todos os atendimentos [Área problema],1338
Requisitos obrigatórios para todos os atendimentos [Círculo de proximidade],1338
Outros comentários,1338


/tmp/ipykernel_19467/3350358904.py:127: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(valor, errors="coerce", dayfirst=True)


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>